# LD comparison — SNPs (GrENE-Net vs production) and SVs (production)

Two questions:

**(A) Are SNP LD-decay curves comparable between two 231-founder panels?**
- GrENE-Net 231-founder SNP-only catalog (`data/vcf/greneNet_final_v1.1.recode.vcf.gz`)
- Our production merged 231-founder pangenome (`pangenie_genotyping/data/merged/founders_231_chr.vcf.gz`) — SNP records only

Same panel size + same ecotypes → SNP-LD curves SHOULD overlay if both catalogs capture the same population structure.

**(B) How does SV LD compare to SNP LD in the production panel?**
- SV anchors (50bp+) in `founders_231_chr.vcf.gz`
- Compared with their SNP/indel/SV neighbors within 1Mb
- Tells us how taggable SVs are by SNPs (relevant for downstream GWAS / haplotype methods)

## Method

All LD r² computed via `preprocess_qc/scripts/compute_ld.py`:
- Filter: biallelic, MAF ≥ 0.05, ≥70% non-missing call rate
- Mixed ploidy handled (cactus haploid → dose 0/2; PanGenie diploid → 0/1/2)
- 5,000 anchor markers per run, neighbors within 1Mb
- Per pair: Pearson r² across the 231-founder dose vector

Run on **Chr1** (largest chrom, most diverse) via `run_ld_chr1.sh` SLURM job.
Outputs in `preprocess_qc/output/ld/`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

LD = Path('../output/ld')
# Glob per source across Chr1-Chr5 and concat with a chromosome column.
sources = {
    'GrENE-Net SNP-only (231)':                'grenenet_snp_chr*.tsv',
    'Production merged (231) — SNP anchors':   'merged_snp_chr*.tsv',
    'Production merged (231) — SV anchors':    'merged_sv_chr*.tsv',
}
ld_data = {}
for name, pat in sources.items():
    parts = []
    for p in sorted(LD.glob(pat)):
        chrom = p.stem.split('_chr')[-1]
        df = pd.read_csv(p, sep='\t')
        df['chrom'] = chrom
        parts.append(df)
    if parts:
        ld_data[name] = pd.concat(parts, ignore_index=True)
        chr_counts = ld_data[name].chrom.value_counts().sort_index().to_dict()
        print(f'  {name}: {len(ld_data[name]):,} pairs across chroms {chr_counts}')
    else:
        print(f'  {name}: NO data yet — wait for ld_one.sh array (60830) to land')


## 1. SNP LD-decay — apples-to-apples between panels

In [ ]:
def bin_decay(df, bins=None, anchor_filter=None, neighbor_filter=None):
    if anchor_filter is not None:
        df = df[df.class1.isin(anchor_filter)]
    if neighbor_filter is not None:
        df = df[df.class2.isin(neighbor_filter)]
    if bins is None:
        bins = np.logspace(0, 6, 25)
    df = df.copy()
    df['bin'] = pd.cut(df.distance, bins=bins, labels=False)
    g = df.groupby('bin').agg(median_r2=('r2','median'),
                              mean_r2=('r2','mean'),
                              q25=('r2', lambda s: s.quantile(0.25)),
                              q75=('r2', lambda s: s.quantile(0.75)),
                              n=('r2','size'))
    g['bin_lo'] = bins[:-1][g.index.astype(int).values]
    g['bin_hi'] = bins[1:][g.index.astype(int).values]
    g['bin_mid'] = np.sqrt(g.bin_lo * g.bin_hi)
    return g

fig, ax = plt.subplots(figsize=(10, 5))
for name, color in [
    ('GrENE-Net SNP-only (231)', '#7f8da6'),
    ('Production merged (231) — SNP anchors', '#1f4e79'),
]:
    if name in ld_data:
        d = bin_decay(ld_data[name], anchor_filter={'SNP'}, neighbor_filter={'SNP'})
        ax.plot(d.bin_mid, d.median_r2, label=f'{name}  (n={d.n.sum():,})', color=color, lw=2)
        ax.fill_between(d.bin_mid, d.q25, d.q75, alpha=0.18, color=color)
ax.set_xscale('log'); ax.set_xlabel('pairwise distance (bp)')
ax.set_ylabel('LD r² (median per distance bin; shaded = IQR)')
ax.set_title('SNP-SNP LD decay — Chr1, MAF≥0.05, 231 founders')
ax.set_ylim(0, 1.0)
ax.legend()
plt.tight_layout(); plt.show()

## 2. SV LD-decay vs SNP LD-decay (production panel only)

How do SVs' LD profiles compare to SNPs at the same panel + chrom?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: anchor=SNP vs neighbor=SNP   AND   anchor=SV vs neighbor=anything
ax = axes[0]
if 'Production merged (231) — SNP anchors' in ld_data:
    d = bin_decay(ld_data['Production merged (231) — SNP anchors'],
                  anchor_filter={'SNP'}, neighbor_filter={'SNP'})
    ax.plot(d.bin_mid, d.median_r2, label='SNP↔SNP', color='#1f4e79', lw=2)
    ax.fill_between(d.bin_mid, d.q25, d.q75, alpha=0.18, color='#1f4e79')
if 'Production merged (231) — SV anchors' in ld_data:
    sv = ld_data['Production merged (231) — SV anchors']
    for cls, color in [('small_sv','#d8b365'), ('medium_sv','#bf812d'), ('large_sv','#8c510a')]:
        d = bin_decay(sv, anchor_filter={cls}, neighbor_filter={'SNP'})
        if len(d) > 0:
            ax.plot(d.bin_mid, d.median_r2, label=f'{cls}↔SNP', color=color, lw=2)
ax.set_xscale('log'); ax.set_xlabel('distance (bp)')
ax.set_ylabel('median r²')
ax.set_title('LD decay — SV anchors vs SNP neighbors')
ax.set_ylim(0, 1)
ax.legend()

# Right: anchor=SV vs neighbor=SV (SV-tagging-SV)
ax = axes[1]
if 'Production merged (231) — SV anchors' in ld_data:
    sv = ld_data['Production merged (231) — SV anchors']
    for cls, color in [('small_sv','#d8b365'), ('medium_sv','#bf812d'), ('large_sv','#8c510a')]:
        d = bin_decay(sv, anchor_filter={cls}, neighbor_filter={'small_sv','medium_sv','large_sv'})
        if len(d) > 0:
            ax.plot(d.bin_mid, d.median_r2, label=f'{cls}↔any SV', color=color, lw=2)
ax.set_xscale('log'); ax.set_xlabel('distance (bp)')
ax.set_ylabel('median r²')
ax.set_title('LD decay — SV anchors vs SV neighbors')
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout(); plt.show()

## 3. r² distribution at fixed distance windows

In [ ]:
windows = [(0, 1_000), (1_000, 10_000), (10_000, 100_000), (100_000, 1_000_000)]
fig, axes = plt.subplots(1, len(windows), figsize=(16, 4), sharey=True)
for ax, (lo, hi) in zip(axes, windows):
    label = f'{lo//1000}-{hi//1000} kb' if hi >= 1000 else f'{lo}-{hi} bp'
    bins = np.linspace(0, 1, 21)
    cohorts = []
    if 'GrENE-Net SNP-only (231)' in ld_data:
        d = ld_data['GrENE-Net SNP-only (231)']
        sub = d[(d.distance >= lo) & (d.distance < hi) & (d.class1=='SNP') & (d.class2=='SNP')]
        cohorts.append(('GrENE-Net SNP', sub.r2, '#7f8da6'))
    if 'Production merged (231) — SNP anchors' in ld_data:
        d = ld_data['Production merged (231) — SNP anchors']
        sub = d[(d.distance >= lo) & (d.distance < hi) & (d.class1=='SNP') & (d.class2=='SNP')]
        cohorts.append(('Production SNP', sub.r2, '#1f4e79'))
    if 'Production merged (231) — SV anchors' in ld_data:
        d = ld_data['Production merged (231) — SV anchors']
        sub = d[(d.distance >= lo) & (d.distance < hi) & d.class1.isin(['small_sv','medium_sv','large_sv'])]
        cohorts.append(('Production SV (any size)', sub.r2, '#d8b365'))
    for name, r2, color in cohorts:
        if len(r2) > 0:
            ax.hist(r2, bins=bins, alpha=0.55, label=f'{name} (n={len(r2):,})', color=color, density=True)
    ax.set_xlabel('r²'); ax.set_title(label); ax.legend(fontsize=8)
axes[0].set_ylabel('density')
plt.tight_layout(); plt.show()

## 4. Median r² as a function of size class (anchor side)

Within the 1-100 kb window, what's the typical r² we can expect with a SNP neighbor?

In [ ]:
rows = []
for src_label, src_key in [
    ('GrENE-Net SNP', 'GrENE-Net SNP-only (231)'),
    ('Production SNP', 'Production merged (231) — SNP anchors'),
    ('Production SV', 'Production merged (231) — SV anchors'),
]:
    if src_key not in ld_data: continue
    d = ld_data[src_key]
    for window_label, lo, hi in [
        ('<1 kb', 0, 1_000),
        ('1-10 kb', 1_000, 10_000),
        ('10-100 kb', 10_000, 100_000),
        ('100kb-1Mb', 100_000, 1_000_000),
    ]:
        for cls in ['SNP','small_indel','small_sv','medium_sv','large_sv']:
            sub = d[(d.distance >= lo) & (d.distance < hi) & (d.class1 == cls)]
            if len(sub) < 10: continue
            rows.append({'source': src_label, 'window': window_label, 'anchor_class': cls,
                         'n_pairs': len(sub), 'median_r2': sub.r2.median()})
table = pd.DataFrame(rows)
if len(table):
    pivot = table.pivot_table(index=['source','anchor_class'], columns='window', values='median_r2').round(3)
    print(pivot)
else:
    print('no data yet — wait for run_ld_chr1.sh to finish')

## 5. Headline summary

After cells 1–4 land, populate the interpretation here:

- Do GrENE-Net and production SNP-LD curves overlay? → if yes, the two SNP catalogs capture the same haplotype structure (sanity check).
- Does SV-LD decay match SNP-LD decay at comparable distance? → if comparable, SVs are taggable by SNPs (good for hapFIRE-style LD-borrowing methods); if much lower, SVs need direct genotyping (which we have via PanGenie).
- Does SV-SV LD show signatures of clustered SV regions / structural-variant hotspots?

The 5,000-anchor sampling is enough to fix the median LD-decay curves on Chr1; for finer per-class breakdowns, re-run with more anchors.